# 03 — Train on Your Own Data

End-to-end walkthrough: load a CSV file, featurize, configure a model,
train, evaluate, and save a checkpoint ready for deployment.

**What you'll learn**
- `CageFusionDataModule.from_csv` — one-call featurization pipeline
- `CageFusionConfig` — set architecture, task type, and label names
- `Trainer` — train with early stopping and best-model checkpointing
- Save the checkpoint + scaler; load back with `CageFusionPipeline`

In [ ]:
import pandas as pd
import torch

from cage_fusion import CageFusionConfig, AutoCageFusion
from cage_fusion.data import CageFusionDataModule
from cage_fusion.training import Trainer, TrainingArguments

## 1. Prepare your CSV

Your CSV must have:
- A `SMILES` column (or pass `smiles_col=` with the actual column name)
- One or more label columns (0/1 for classification, floats for regression)

Below we generate a tiny synthetic example.  Replace with your own file.

In [ ]:
# ── Create a toy CSV ─────────────────────────────────────────────────────────
toy_data = [
    {"SMILES": "CC(=O)Oc1ccccc1C(=O)O",                         "active": 0, "toxic": 0},
    {"SMILES": "c1ccc2ccccc2c1",                                  "active": 0, "toxic": 0},
    {"SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",                   "active": 1, "toxic": 0},
    {"SMILES": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",             "active": 1, "toxic": 0},
    {"SMILES": "O=C(O)c1ccccc1O",                                  "active": 0, "toxic": 0},
    {"SMILES": "C1CCCCC1",                                         "active": 0, "toxic": 0},
    {"SMILES": "CCO",                                              "active": 0, "toxic": 0},
    {"SMILES": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                      "active": 1, "toxic": 0},
    {"SMILES": "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3","active": 1, "toxic": 1},
    {"SMILES": "O=C1c2ccccc2C(=O)c3ccccc13",                       "active": 0, "toxic": 1},
]

df = pd.DataFrame(toy_data)
df.to_csv("/tmp/my_compounds.csv", index=False)

print(f"Dataset: {len(df)} rows")
print(df.head())

## 2. Build the data module

`from_csv` handles:
- Train / val (/ test) splitting
- RDKit auxiliary feature computation + scaler fitting
- ChemBERTa tokenization and embedding
- HDF5 streaming feature caching

In [ ]:
LABEL_COLS = ["active", "toxic"]
CHECKPOINT_DIR = "/tmp/cage_fusion_custom"

dm = CageFusionDataModule.from_csv(
    csv_path="/tmp/my_compounds.csv",
    label_cols=LABEL_COLS,
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
    val_split=0.15,
    test_split=0.10,
    cache_dir="/tmp/cage_features",
    batch_size=8,      # small for this toy example
)

print("Label names :", dm.label_names)
print("Train batches:", len(dm.train_loader))
print("Val batches  :", len(dm.val_loader))
print("Test batches :", len(dm.test_loader) if dm.test_loader else "None")

## 3. Configure the model

In [ ]:
config = CageFusionConfig(
    num_labels=len(dm.label_names),
    model_task="classification",
    label_names=dm.label_names,
    attn_mode="cross",          # paper's default — bidirectional gated co-attention
    use_fg_prompt=True,
    hidden_size=128,
)

print(config)

## 4. Build the model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoCageFusion.from_config(config).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")

## 5. Train

In [ ]:
args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    checkpoints_dir=CHECKPOINT_DIR,
    num_epochs=30,
    batch_size=8,
    learning_rate=3e-4,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

history = trainer.train()
print("Training complete.  Best val AUC:", max(history["val_auc"]))

## 6. Plot training history

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs, history["val_auc"])
axes[1].set_title("Val ROC-AUC")

axes[2].plot(epochs, history["val_mcc"])
axes[2].set_title("Val MCC")

plt.tight_layout()
plt.show()

## 7. Save the scaler and config

In [ ]:
import os

# The Trainer already saved best_model.pt to CHECKPOINT_DIR.
# Save the scaler (required by the pipeline) and the config.
dm.save_scaler(CHECKPOINT_DIR)
config.save_pretrained(CHECKPOINT_DIR)

print("Files saved:")
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    print(" ", f)

## 8. Load back with the pipeline

In [ ]:
from cage_fusion import CageFusionPipeline

pipe = CageFusionPipeline.from_pretrained(CHECKPOINT_DIR)

# Quick sanity check
print(pipe("CC(=O)Oc1ccccc1C(=O)O"))